In [ ]:
from pathlib import Path 
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 

pd.options.display.float_format = (lambda x: f'{x:,.2f}')
pd.options.display.max_columns = None 

parent_dir = Path.cwd()
data_path_1 = parent_dir/...  # can do .txt as well as .csv
data_path_2 = parent_dir/...

cols = [col1, col2, ...]     # if data doesn't have labelled columns, can feed this in below

df_all_1 = pd.read_csv(data_path_1, sep = '|', header = None, names = cols, dtype = DICT, parse_dates = ['date_col'])   # if names provided then header is None by default
df = ...
df.columns.str.replace(' ', '_')    # if columns are provided but with whitespace, then can replace with underscores 

In [ ]:
# if 'date_col' contains e.g. different UTC offsets, then parse_dates might not work. Can do following:

df['date_col'] = pd.to_datetime(df['date_col'], utc = True)     # converts to UTC (which does not observe daylight savings). If 'date_col' not dtype str, use df['date_col'].astype(str)
df['date_col'] = df['date_col'].dt.tz_convert('Region')         # converts to local time of 'Region', including the offset from UTC
df['day_of_week'] = df['date_col'].dt.day_of_week                   

df['date_col'] = df['date_col'].dt.tz_localize(None)            # if you want to remove UTC offset 
df['date_col'] = df['date_col'] + pd.Timedelta(hours = 5)       # if you want to manually add 5 hours

# sometimes it may not be clear how pandas should read the strings in 'date_col' to convert them into datetime, e.g. if presented as 'YYYYMM' then the following would work: 

df['date_col'] = pd.to_datetime(df['date_col'], format = '%Y%m%')       # day defaults to first of month (and displayed), time defaults to midnight (but not displayed)
df['date_col'].dt.to_period('M')                                        # assuming previous line has been done, this forgets day and time information and retains only month-year; becomes period[M] dtype
df_eda = df[df['date_col'] <= pd.Period('YYYY-MM', freq = 'M')] 

In [ ]:
from sklearn.model_selection import train_test_split 

train_df, test_df = train_test_split(df, test_size = 0.2, random_state = 42, shuffle = True)    # retains indexing of df (but rows shuffled)
df_eda = train_df.copy()

train_df = df[df['date_col'].dt.year <= 2020]   # or e.g:
train_df = df[df['date_col'] <= pd.Period('YYYY-MM', freq = 'M')] 
df_eda = train_df.copy()

In [ ]:
df_1 = df_all_1.drop(columns = LIST)

df.describe(percentiles = LIST)

df.isna().sum()                 # df.isna() returns boolean DataFrame of same size, with True for missing values. Doing .sum() then sums over axis = 0 (by default), i.e. returns number of missing values for each column 
df['col_name'].isna().sum()     
df[df['col_name'].isna()]       # return subset of df consisting of rows for which the 'col_name' entry is missing
df[df.isna().any(axis = 1)]     # return subset of df consisting of rows for any entry is missing; .any(axis = 1) returns True for rows where at least one entry in that row is True

df.duplicated().sum()                           # df.duplicated() returns boolean Series asserting whether each row is a duplicated. Doing .sum() then finds number of duplicate rows 
df.duplicated(subset = 'date_col').sum()        # same as above but considers to rows duplicates if they have the same 'date_col' entries. Can replace 'date_col' with list of column names

df.loc[df['col_name'] >= outlier_value, ['col_1_to_display', 'col_2_to_display']].sort_values(by = 'col_name', ascending = False)       # to look at rows with extreme values for 'col_name' 

In [ ]:
df = df.sort_values(['col_1', 'col_2'])                 # sort by 'col_1' first then 'col_2'; index is not reset 

df.insert(3, 'new_col', df[LIST_OF_COLS].sum(axis = 1)) # alternative to creating new feature directly, specifying column number of new column

df['shifted_col'] = df.groupby('id_col')['col_to_shift'].shift(1)

agg_quantities = df.groupby('period').agg(any_name_1 = ('col1_to_aggregate', 'sum'), ...)         # creates DataFrame with columns 'any_name_1', 'any_name_2' and indexed by 'period'         

occurs = df['col_name'].isin(['option_1', 'option_2'])      # boolean Series with same index as df, True if a row's value of 'col_name' belongs to ['option_1', 'option_2']
ever_occurs = occurs.groupby(df['id']).cummax()             # occurs does not have 'id' column but does have same indices as df; cummax() is equal to 1 if the thing occurs for any row with that id 

df['new_col'] = df['col_1'].where(condition, replacement_value)      # where condition is True, 'new_col' = 'col_1', otherwise it is set to replacement_value 

df_clean = df.loc[valid]  

In [ ]:
df = df_1.merge(df_2, on = 'common_col', how = 'left', validate = 'many_to_one')                 # in this case we assume df_1 has multiple of the same entries in 'common_col' and df_2 only has one. Index automatically reset

# suppose e.g. df_3 has arisen from aggregation on some grouping, say by period, yileding the average value of 'col_1' for each fixed value of 'period'. Then we can add a new column 'new_col' in df with this average value for each fixed value of 'period':

df_3 = df.groupby('period')['col_1'].mean()                                                      # values of 'period' are then the indices the df_3 Series
df = df.join(df_3.rename('new_col'), on = 'period', how = 'left', validate = 'many_to_one')      # relies on 'period' being a column in df, and its values being indices in df_3 
df['new_col'] = df.groupby('period')['col_1'].transform('mean')                                  # alternative

In [ ]:
df['col_name'].value_counts(dropna = False)         # returns Series counting number of occurrences for each unique value in the column 'col_name' 
df['col_name'].value_counts(dropna = False).plot.bar(...)       

# can replace df with df.loc['YYYY-MM-DD':'yyyy-mm-dd'] to get plots for smaller range of dates, assuming index are set to dates (using e.g. .set_index())

df.hist(bins = 50, ...)                                             # automatically selects only numerical columns     
df.plot(y = ['col_1', 'col_2'], secondary_y = 'col_1', ...)         
df.plot.bar(y = ['col_1', 'col_2'], legend = True, ...)             # index used for x-axis
df.plot.scatter(x = 'col_1', y = 'col_2', alpha = 0.1, s = 5, ...)
df.plot.scatter(x = 'col_1', y = 'col_2', c = 'target', cmap = 'viridis', colorbar = True, ...)
df.boxplot(column = 'target', by = 'cat_col', ...)                  # no title argument but can do xlabel and ylabel

df_weekly = df.set_index('date_col')['col_name'].resample('W').mean()    # creates Series consisting of average values of 'col_name' for each week; easier to plot than higher frequency data 

In [ ]:
df_numerical = df.select_dtypes(include = 'number')         # only numerical columns of df 

corr_matrix = df_numerical.corr()       # could also do numeric_only = True if non-numerical columns present
target_corrs = corr_matrix['target'].drop('target').sort_values(ascending = False)     # returns Series of correlations with target variable; requires all columns in df_numerical to be numerical 
upper_tri = np.triu(np.ones(corr_matrix.shape, dtype = bool), k = 1)
corrs = corr_matrix.where(upper_tri).rename_axis(index = 'variable_1', columns = 'variable_2').stack().rename('correlation')   # without renaming, could assign corrs.columns = ['variable_1', 'variable_2', 'correlation']
corrs = corrs.sort_values(by = 'correlation', key = lambda x: abs(x), ascending = False) 

df['target_band'] = pd.qcut(df['target'], q = 3, labels = ['small', 'medium', 'large'])        # to explore how correlations between 'feature' and others depends on size of 'target'
for band in ['small', 'medium', 'large']:
    subset = df[df['target_band'] == band]
    corr_col = subset.corr(numeric_only = True)['feature'].rename(f'Correlation when target is {band}')
    corrs = corrs.join(corr_col)

df['time_band'] = pd.cut(df['hour'], bins = [0, 6, 12, 18, 24], labels = ['00-06', '06-12', '12-18', '18-24'], right = False)       # to explore how correlation between 'feature' and others depends on hour of day
for band in ['00-06', '06-12', '12-18', '18-24']...

To add a table in markdown: 

| Finding | Initial modelling decision or experiment |
|---|---|
| Finding 1 | Decision 1 |
| Finding 2 |  Decision 2 |

In [ ]:
target = 'target'
target_2 = 'target_2'       # assume binary valued for logistic regression model 
candidate_features = ['col_1', 'col_2', ...]

X_train = train_df[candidate_features]      # note some features may have been introduced to eda_df but not train_df
y_train = train_df[target]

train_df = df_clean[df_clean['date_col'] <= ...]
val_df = ...

X_train = train_df[candidate_features]
y_train = train_df[target]
y_train_class = train_df[target_2] 

X_val = val_df[candidate_features] ...

numeric_cols = X_train.select_dtypes(include = 'numeric').columns.to_list()
categorical_cols = [col for col in candidate_cols if col not in numeric_cols]

In [ ]:
from sklearn.model_selection import cross_validate, KFold       # cross_validate takes as input cv (the cross-validation generator, below cv = KFold(...)) and a scoring dict

cv = KFold(n_splits = 5, shuffle = True, random_state = 42)
scoring = {'RMSE': 'neg_root_mean_squared_error', ...} 
fold_scores = cross_validate(model, X_train, y_train, cv = cv, scoring = scoring, error_score = 'raise')       # model such as ols_baseline (see below) is what implements fit

# fold_scores returns dict with keys 'test_XXXX' (where 'XXXX' are the keys in scoring) and values given by the array of XXXX scores for each held-out fold 

def evaluate_model(model, X, y, name):
    fold_scores = cross_validate(model, X, y, cv = cv, scoring = scoring, error_score = 'raise')
    fold_scores = pd.DataFrame({f'{name}_RMSE': -fold_scores['test_rmse'], ...})
    fold_scores.index = ...
    scores = pd.concat([fold_scores, fold_scores.agg(['mean', 'std'])])     
    return scores  

# very basic baseline model  predicts the mean of the training observations:

from sklearn.dummy import DummyRegressor 
dummy_scores = evaluate_model(DummyRegressor(strategy = 'mean'), X_train, y_train, 'dummy_regressor')

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer 
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from sklearn.base import clone 

numeric_preprocessing = Pipeline([                      # takes list consisting of tuples (name of step, estimator)
    ('impute', SimpleImputer(strategy = 'median')),
    ('scalar', StandardScaler())
])

preprocessing = ColumnTransformer([                     # takes list consisting of tuples (name of step, transformer, columns)
    ('numeric', numeric_preprocessing, numeric_cols),
    ('categorical', OneHotEncoder(handle_unknown = 'ignore'), categorical_cols)
])

ols_baseline = Pipeline([
    ('preprocessing', preprocessing),
    ('model', LinearRegression())
])

forest_baseline = ...
classifier_baseline = ...

classifier_baseline.fit(X_train, y_train_class)... 

In [ ]:
ols_initial_predictions_train = ols_baseline.predict(X_train)
ols_initial_predictions_val = ...

classifier_initial_predictions_train = classifier_baseline.predict_proba(X_train)[:,1]
classifier_initial_predictions_val = ...

# if our pipelines predict an initial quantity that must transformed into a target prediction, then might need to aggregate. E.g. suppose we are making a daily or monthly prediction, then might use something like:

def target_predictions(data, initial_predictions = None)        # initial_predictions = None could be used for baseline model predicting yesterday's target. Data will be e.g. train_df or val_df
    if initial_predictions is None:
        ...
        daily_data['predicted_target'] = daily_data['actual_target'].shift(1)
    
    daily_data = data.groupby('date_col').agg(summed_quantity_1 = ('col_with_quantity_1s', 'sum'),
                                              summed_quantity_2 = ('col_with_quantity_2s', 'sum'))      # remember index will be 'date_col' values 
    daily_data['actual_target'] = ...
    daily_data['predicted_target'] = ...
    return daily_data 

def prediction_scores(daily_data):
    actual_target = daily_data['actual_target']
    predicted_target = ...
    return {'RMSE': root_mean_squared_error(actual_target, predicted_target), ...}

In [ ]:
previous_month_target_predictions_train = target_predictions(df_clean, None).loc['YYYY-MM':'yyyy-mm']
previous_month_target_predictions_val = ...

ols_target_predictions_train = target_predictions(train_df, ols_initial_predictions_train)
ols_target_predictions_val = ...

training_scores = {
    'Previous month (training)': prediction_scores(previous_month_target_predictions_train),        # dictionary where each value is also a dictionary 
    ...}
validation_scores = ...

display(pd.DataFrame(training_scores).T)        # without .T, keys of outer dictionary (e.g. 'Previous month (training)') will be column names, and keys of inner dictionary ('RMSE', 'MAE', 'Bias') will be row indices 

# concatenate various predictions and actual target, and plot

validation_predictions = pd.concat([previous_month_target_predictions_val['actual_target'],     # axis = 1 means columns side-by-side; all dataframes here have same index (period), so concatenation will too
                                    previous_month_target_predictions_val['predicted_target'],  # more generally, concat keeps union of all indices and uses NaN for missing values 
                                    ols_target_predictions_val['predicted_target'], ...], axis = 1)

validation_predictions_columns = ['Actual target', ...]
validation_predictions.plot(y = ['col_1', 'col_2', ...], color = ['black', 'C0', ...])

In [ ]:
# suppose we import a dataset with multiple values per month and we want to aggregate these into one value per month:

external_df = pd.read_csv('file_name.csv')
external_df['year_month'] = pd.to_datetime(external_df['date_col'].astype(str)).dt.to_period('M')
monthly_external_df = external_df.groupby('year_month')['new_variable'].mean()
prev_monthly_external_df = monthly_external_df.shift(1)

df_clean = df_clean.join(prev_monthly_external_df.rename('prev_monthly_external_df'), on = 'year_month', how = 'left', validate = 'many_to_one')
df_clean['new_col'] = ...

df_monthly_shifted = df_monthly['target'].shift(1)      # where df_monthly is some aggregated groupby on 'year_month' (or whatever period), and hence has 'year_month' as index (unless reset)
df_clean = df_clean.join(df_monthly_shifted.rename('prev_target'), on = 'year_month', ...)

In [ ]:
# re-run pipelines, model fits and predictions to include new features, then compare scores 

candidate_features_new = candidate_features + ['new_col', 'prev_target']
X_train/val_new = df_clean.loc[train_df.index, candidate_features_new]      # new features haven't been added to train_df or val_df, so take subset of df_clean
numeric_cols_new = numeric_cols + ...
preprocessing_new = ...                                                     # ColumnTransformer with ('numeric', clone(numeric_preprocessing), numeric_cols_new)
ols_new = clone(ols_baseline).set_params(preprocessing = clone(preprocessing_new))
...

In [ ]:
cv_years = [2021, 2022, 2023]   # for expanding window
cv_folds = []

for year in cv_years:
    train = df_clean[...]
    val = df_clean[...]
    cv_folds.append((train, val))

def evaluate_cv(model, classifier = False):
    monthly_results = []
    fold_results = []

    for (train, val) in cv_folds:
        # follow same procedure as before, leading to:
        model_target_predictions = target_predictions(val, model_initial_predictions)
        monthly_results.append(model_target_predictions)
        fold_results.append(prediction_scores(model_target_predictions))
    
    all_model_results = pd.concat(monthly_results)      # concatenates vertically
    overall_scores = prediction_scores(all_model_results)

    return {'2021 RMSE': fold_results[0]['RMSE'], ..., 'CV RMSE': overall_scores['RMSE'], ...}, all_model_results


cv_results = pd.DataFrame({name: evaluate_cv(model, classifier)[0] for name, (model, classifier) in evaluate_cv_inputs.items()}).T      # outer keys (i.e. name) will be col names and inner keys ('2021 RMSE' etc.) will be row indices

prev_month_target_preds_cv = target_predictions(df_clean, None).loc['2021-01':'2023-12']
cv_results.loc['Previous month', 'CV RMSE'] = prediction_scores(prev_month_target_preds_cv)['RMSE'] ...
for year in cv_years:
    restricted_predictions = prev_month_target_preds_cv.loc[f'{year}...']

# hyperparameter grid search:
forest_comparisons_rmse = pd.DataFrame(index = max_depths, columns = min_samples_leaf, dtype = float)
forest_comparisons_rmse.index.name = 'max_depth (RMSE)'
forest_comparisons_rmse.columns.name = 'min_samples_leaf (RMSE)'

# simple ensemble:
ensemble_predictions = (forest_cv_predictions[['actual_target', 'predicted_target']] + prev_month_target_preds_cv[['actual_target', 'predicted_target']]) / 2


### Diagnostics

In [ ]:
from sklearn.model_selection import cross_val_predict       # returns actual predictions too 

ols_predictions = cross_val_predict(ols_baseline, X_train, y_train, cv = cv, n_jobs = 1)
ols_predictions = pd.DataFrame(ols_predictions, index = X_train.index)
ols_predictions = ols_predictions.join(y_train)                                 # could also look directly at residuals y_train.to_numpy() - ols_predictions
ols_predictions.columns = ['predicted_target', 'actual_target']     

ax = ols_predictions.plot.scatter(x = 'actual_target', y = 'predicted_target', ...)
ax.plot([0, x_range], [0, y_range], 'r---')

diagnostics = X_train.copy()    # to find predictions with largest residuals
diagnostics['actual_target'] = y_train 
diagnostics['predicted_target'] = ols_predictions['predicted_target']
diagnostics['residual'] = diagnostics['actual_target'] - diagnostics['predicted_target']
display(diagnostics.loc[diagnostics['residual'].abs().nlargest(10).index, cols])

ols_predictions['squared_error'] = (ols_predictions['actual_target'] - ols_predictions['predicted_target'])**2      # similarly could compute RMSE over certain rows we suspect of being most damaging to predictions
top_1pct = int(np.ceil(0.01 * len(ols_predictions)))
fraction = np.sort(ols_predictions['squared_error'])[-top_1pct:].sum() / ols_predictions['squared_error'].sum()

In [ ]:
from sklearn.inspection import permutation_importance 

train_idx, val_idx = next(cv.split(X_train))    # look at first fold only 
forest = clone(forest_baseline)
forest.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])

importance = permutation_importance(forest, X_train.iloc[val_idx], y_train.iloc[val_idx], scoring = 'neg_root_mean_squared_error', n_repeats = 5, random_state = 42)
importance_df = pd.Series(importance.importances_mean, index = X_train.columns).sort_values()
importance_df.plot.barh(figsize=(7, 5))

from sklearn.inspection import PartialDependenceDisplay

PartialDependenceDisplay.from_estimator(forest, X_train.iloc[train_idx], features=['feature_1'], grid_resolution=30)
# by default only considers feature values between the 5th and 95th percentiles. Bold vertical lines are rug marks, placed at each decile of that feature's distribution
plt.tight_layout()...